# Tema 6 · Selección de atributos

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

La **Cooperativa Triguera de San Ignacio** reúne a pequeños productores de trigo. Durante una temporada anotaron **todo** lo que pudieron de 160 potreros: 20 datos por potrero, desde el nitrógeno aplicado hasta la edad del agricultor o la antigüedad del tractor. Ahora quieren un modelo que prediga el **rendimiento** (quintales por hectárea), pero medir tantas cosas les cuesta caro y no saben cuáles sirven.

> **❓ La pregunta del cliente:** ¿Qué datos de cada potrero sirven de verdad para predecir el rendimiento, y cuáles pueden dejar de medir?

## 🗂️ Los datos: `t6_trigo.csv` (160 potreros con 20 atributos y el rendimiento de trigo)

| Columna | Qué significa |
|---|---|
| `numero_potrero` | número del potrero |
| `nitrogeno_kg_ha` | nitrógeno aplicado, en kg por hectárea |
| `nitrogeno_sacos` | el mismo nitrógeno, contado en sacos de 25 kg por hectárea |
| `lluvia_mm` | lluvia de la temporada, en milímetros |
| `lluvia_pulgadas` | la misma lluvia, en pulgadas |
| `dia_siembra` | día del año en que se sembró (120 = fines de abril) |
| `ph_suelo` | pH del suelo (acidez) |
| `distancia_bodega_km` | distancia del potrero a la bodega de la cooperativa |
| `edad_agricultor` | edad del socio, en años |
| `hectareas` | superficie del potrero |
| `pendiente_pct` | pendiente del terreno, en % |
| `anios_cultivo` | años que lleva cultivándose |
| `altitud_m` | altitud, en metros |
| `temperatura_bodega` | temperatura media de la bodega del socio |
| `horas_maquinaria` | horas de maquinaria usadas |
| `trabajadores` | personas que trabajaron el potrero |
| `distancia_camino_m` | distancia al camino, en metros |
| `antiguedad_tractor` | años del tractor usado |
| `largo_potrero_m` | largo del potrero, en metros |
| `visitas_tecnico` | visitas del técnico en la temporada |
| `rendimiento_qq_ha` | **rendimiento de trigo, en quintales por hectárea** ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_regression

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t6_trigo.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 160 filas y 21 columnas.

## 1. Conocer los datos

📘 Cada fila es un **potrero** de trigo de un socio de la cooperativa. Hay **20 atributos** (todo lo que se anotó de
cada potrero, sirva o no) y el **rendimiento** en quintales por hectárea (qq/ha), que es lo que se quiere predecir.
La celda cuenta potreros y atributos.

In [ ]:
datos.head()

In [ ]:
X = datos.drop(columns="rendimiento_qq_ha")      # los 20 atributos
y = datos["rendimiento_qq_ha"]                      # lo que se quiere predecir
print("Potreros:", len(datos), "· Atributos:", X.shape[1])
# ✅ Debe mostrar: Potreros: 160 · Atributos: 20

## 2. Separar entrenamiento y prueba

📘 La cooperativa tiene **pocos potreros** (160). La mitad se usa para **entrenar** y la otra mitad para **probar**
con potreros que el modelo no vio. Con pocos datos y muchos atributos aparece el riesgo que estudia este tema.

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.5, random_state=2026)
print("Entrenamiento:", len(X_ent), "potreros · Prueba:", len(X_pru), "potreros")
# ✅ Debe mostrar: Entrenamiento: 80 potreros · Prueba: 80 potreros

## 3. Método de filtro: ¿qué tanto se relaciona cada atributo con el rendimiento?

📘 La **correlación** mide qué tan juntas se mueven dos columnas: cerca de **1** (o −1) es una relación fuerte;
cerca de **0**, ninguna. Se calcula la de cada atributo con el rendimiento (en valor absoluto, porque una relación
negativa también sirve) y se ordenan de mayor a menor. Esto es un **filtro**: mira cada atributo **por separado**,
antes de construir el modelo.

In [ ]:
correlacion = X_ent.corrwith(y_ent).abs().sort_values(ascending=False)

plt.figure(figsize=(7, 5))
plt.barh(correlacion.index[::-1], correlacion.values[::-1], color="#1f5c3d")
plt.xlabel("Correlación con el rendimiento (valor absoluto)")
plt.title("¿Qué atributos se relacionan con el rendimiento?")
plt.show()

print("Los 6 más relacionados:", ", ".join(correlacion.index[:6]))
print("¿Cuántos atributos tienen correlación menor que 0,25?", (correlacion < 0.25).sum())
# ✅ Debe mostrar: Los 6 más relacionados: nitrogeno_kg_ha, nitrogeno_sacos, lluvia_pulgadas, lluvia_mm, ph_suelo, dia_siembra  /  ¿Cuántos atributos tienen correlación menor que 0,25? 14

### ✍️ ¿Qué atributos se relacionan con el rendimiento y cuáles parecen ruido? ¿Les parece lógico, pensando en cómo crece el trigo?

_Su respuesta:_

## 4. Atributos redundantes: copias que no aportan

📘 Dos atributos son **redundantes** cuando dicen lo mismo con otras unidades. La celda mide la correlación
**entre** la lluvia en milímetros y en pulgadas, y entre el nitrógeno en kg/ha y en sacos.

In [ ]:
print(f"Lluvia en mm vs. en pulgadas: {X_ent['lluvia_mm'].corr(X_ent['lluvia_pulgadas']):.2f}")
print(f"Nitrógeno en kg/ha vs. en sacos: {X_ent['nitrogeno_kg_ha'].corr(X_ent['nitrogeno_sacos']):.2f}")
# ✅ Debe mostrar: Lluvia en mm vs. en pulgadas: 1.00  /  Nitrógeno en kg/ha vs. en sacos: 0.99

### ✍️ ¿Qué significa una correlación tan alta entre dos atributos? Si ya tienen la lluvia en milímetros, ¿qué aporta la lluvia en pulgadas?

_Su respuesta:_

## 5. ¿Más atributos es mejor? Todos contra los 4 elegidos

📘 Se entrenan dos regresiones lineales: una con **los 20 atributos** y otra solo con los **4 elegidos** (los
relacionados con el rendimiento, sin las copias). El **R²** dice qué parte de la variación del rendimiento explica
el modelo: 1 es perfecto y 0 es no explicar nada. Se mide en entrenamiento y en prueba.

In [ ]:
elegidos = ["nitrogeno_kg_ha", "lluvia_mm", "dia_siembra", "ph_suelo"]

modelo_todos = LinearRegression().fit(X_ent, y_ent)
modelo_elegidos = LinearRegression().fit(X_ent[elegidos], y_ent)

print(f"Con los 20 atributos → R² entrenamiento: {modelo_todos.score(X_ent, y_ent):.2f} · R² prueba: {modelo_todos.score(X_pru, y_pru):.2f}")
print(f"Con los 4 elegidos   → R² entrenamiento: {modelo_elegidos.score(X_ent[elegidos], y_ent):.2f} · R² prueba: {modelo_elegidos.score(X_pru[elegidos], y_pru):.2f}")
# ✅ Debe mostrar: Con los 20 atributos → R² entrenamiento: 0.83 · R² prueba: 0.69  /  Con los 4 elegidos   → R² entrenamiento: 0.77 · R² prueba: 0.79

### ✍️ ¿Qué modelo explica mejor los potreros de entrenamiento y cuál predice mejor los de prueba? ¿Por qué el modelo con 20 atributos empeora en prueba si tiene «más información»?

_Su respuesta:_

## 6. Selección automática: SelectKBest

📘 `SelectKBest(f_regression, k=K)` es un **filtro automático**: calcula para cada atributo qué tan fuerte es su
relación con el rendimiento y se queda con los **K mejores**. Después se entrena la regresión solo con esos K.
La pregunta es **cuántos** conviene usar.

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
K = 4        # ✏️ pruebe con 1, 2, 4, 6, 10 y 20 (al terminar, deje 4)

selector = SelectKBest(f_regression, k=K).fit(X_ent, y_ent)
columnas = list(X_ent.columns[selector.get_support()])
modelo_k = LinearRegression().fit(X_ent[columnas], y_ent)
print("Eligió:", ", ".join(columnas))
print(f"K = {K} → R² entrenamiento: {modelo_k.score(X_ent[columnas], y_ent):.2f} · R² prueba: {modelo_k.score(X_pru[columnas], y_pru):.2f}")

**✏️ Anoten aquí sus resultados:**

| K (atributos) | R² entrenamiento | R² prueba | ¿Cuáles eligió? (si son pocos) |
|:-:|:-:|:-:|---|
| 1 | | | |
| 2 | | | |
| 4 | | | |
| 6 | | | |
| 10 | | | |
| 20 | | | |

### ✍️ Con K = 4, ¿qué atributos eligió SelectKBest? ¿Son los mismos 4 que eligieron ustedes en la sección 5? ¿Por qué el filtro se equivocó y qué K resultó mejor en prueba?

_Su respuesta:_

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para la Cooperativa Triguera de San Ignacio: ¿qué datos conviene seguir midiendo en cada potrero, cuáles pueden dejar de anotar y qué ganan con eso?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema6.ipynb`, junto con el informe en PDF.